In [2]:
import pandas as pd
import plotly.express as px  # fixme unused
import plotly.io as pio
import plotly.graph_objects as go

# ==== CONSTANTS ====
INPUT_FILE = "in/paper_figures_dataframes.xlsx"
FIG_6_DATA = "in/Figure6_data.csv"
PLOTS_DIR = "out/paper-plots/"

FONT_FAMILY = "Arial, sans-serif"
FONT_SIZE = 26
FONT_COLOR = "#222222"

pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.font.family = FONT_FAMILY
pio.templates["plotly_white"].layout.font.size = FONT_SIZE
pio.templates["plotly_white"].layout.font.color = FONT_COLOR
DEFAULT_WIDTH = 1500
DEFAULT_HEIGHT = 800


def apply_fig_defaults(fig, gridX=True, gridY=True):
    fig.update_layout(
        width=DEFAULT_WIDTH,
        height=DEFAULT_HEIGHT,
        font=dict(
            family=FONT_FAMILY,
            size=FONT_SIZE,
            color=FONT_COLOR,
        ),
        margin=dict(l=30, r=30, t=50, b=0),
        showlegend=True,
        legend=dict(
            title="", # no title for legend
            orientation="h",  # horizontal legend
            x=0.5,  # centered horizontally
            y=-0.25,  # just below the plot area
            xanchor="center",  # anchor x at the center
            yanchor="top",  # anchor y at the top (so it sits just below plot)
        ),
        plot_bgcolor="white",
    )

    # Grid
    if gridX:
        fig.update_xaxes(
            showgrid=True,
            gridcolor="#bbbbbb",
            gridwidth=1,
            ticks="outside",
            ticklen=5,
            linewidth=1,
            linecolor="#222",
        )
    if gridY:
        fig.update_yaxes(
            showgrid=True,
            gridcolor="#bbbbbb",
            gridwidth=1,
            ticks="outside",
            ticklen=5,
            linewidth=1,
            linecolor="#222",
        )

    return fig


def export_fig(fig, filename, w=DEFAULT_WIDTH, h=DEFAULT_HEIGHT):
    fig.write_image(
        PLOTS_DIR + filename + ".pdf",
        format="pdf",
        width=w,
        height=h,
    )
    fig.write_image(
        PLOTS_DIR + filename + ".svg",
        format="svg",
        width=w,
        height=h,
    )
    print(
        f"Figure saved to {PLOTS_DIR + filename}. width={DEFAULT_WIDTH}, height={DEFAULT_HEIGHT}."
    )


COLOR_THEME = px.colors.qualitative.Vivid
COLOR_MAP = {
    "Waste": COLOR_THEME[9],
    "Biomass": COLOR_THEME[3],
    "Lignite": COLOR_THEME[1],
    "Natural gas": COLOR_THEME[10],
    "Geothermal": COLOR_THEME[0],
    "Nuclear": COLOR_THEME[4],
    "Mineral oil": COLOR_THEME[8],
    "Solar": COLOR_THEME[6],
    "Other": COLOR_THEME[9],
    "Industrial gas": COLOR_THEME[10],
    # "Hard coal": COLOR_THEME[10],
    "Hard coal": "#333333",  # black for hard coal
    "Hydropower": COLOR_THEME[7],
    "Wind": COLOR_THEME[2],
    "Hydrogen": COLOR_THEME[5],
    "Other renewables": COLOR_THEME[10],
    # Additional categories for hydrogen production
    "Oil": COLOR_THEME[8],
    "Natural Gas": COLOR_THEME[10],
    "Photovoltaics": COLOR_THEME[6],
    "Hydro": COLOR_THEME[7],
}
PATTERN_MAP = {
    "Waste": "/",
    "Biomass": "\\",
    "Lignite": "x",
    "Natural gas": "|",
    "Geothermal": "-",
    "Nuclear": "+",
    "Mineral oil": ".",
    "Solar": "/",
    "Other": "\\",
    "Industrial gas": "|",
    "Hard coal": "x",
    "Hydropower": "-",
    "Wind": "+",
    "Hydrogen": ".",
    "Other renewables": "/",
    "Oil": "\\",
    "Natural Gas": "|",
    "Photovoltaics": "-",
    "Hydro": "+",
}

LINE_DASH_MAP = {
    "Waste": "solid",
    "Biomass": "solid",
    "Lignite": "solid",
    "Natural gas": "solid",
    "Geothermal": "solid",
    "Nuclear": "solid",
    "Mineral oil": "solid",
    "Solar": "dashdot",
    "Other": "dashdot",
    "Industrial gas": "dash",
    "Hard coal": "solid",
    "Hydropower": "longdash",
    "Wind": "longdash",
    "Hydrogen": "dot",
    "Other renewables": "dot",
    # Additional categories for hydrogen production
    "Oil": "solid",
    "Natural Gas": "solid",
    "Photovoltaics": "dashdot",
    "Hydro": "longdash",
}

# COLOR_MAP = {
#     "Waste": "darkslategray",
#     "Biomass": "forestgreen",
#     "Lignite": "saddlebrown",
#     "Natural gas": "lightskyblue",
#     "Geothermal": "darkorange",
#     "Industrial gas": "lightgray",
#     "Nuclear": "mediumvioletred",
#     "Mineral oil": "dimgray",
#     "Solar": "gold",
#     "Other": "olive",
#     "Hard coal": "black",
#     "Hydropower": "royalblue",
#     "Wind": "lightblue",
#     "Hydrogen": "cyan",
#     "Other renewables": "pink",

#     # Additional categories for hydrogen production
#     "Oil": "darkred",
#     "Natural Gas": "lightskyblue",
#     "Photovoltaics": "gold",
#     "Hydro": "royalblue",
# }


def get_color_sequence(keys):
    """
    Return color sequence for a list of categories, based on COLOR_MAP.
    If a key is missing, fallback to a default color (gray).
    """
    return [COLOR_MAP.get(k, "red") for k in keys]

In [3]:
# Import data from the Excel file

sheets_dict = pd.read_excel(INPUT_FILE, sheet_name=None)  # None = read all sheets as dict

print("Available sheets:", list(sheets_dict.keys()))

Available sheets: ['Figure 1', 'Figure 2', 'Figure 3', 'Figure 4', 'Figure 5', 'Figure 6', 'Figure 7', 'Figure 8']


In [4]:
# Figure 1: Historical electricity costs in €/kwh for 2007 to 2024
# Excel: git:/out/prices/eurostat_electricity_prices_annual.xlsx

fig1_df = sheets_dict["Figure 1"]
fig1_df = fig1_df.set_index("year")
fig1_df = fig1_df.sort_index(ascending=True)

fig = go.Figure()
fig.add_bar(
    x=fig1_df.index,
    y=fig1_df["price"],
    name="Electricity Price [€/kWh]", 
    marker_color=COLOR_THEME[7],
    text=fig1_df["price"].round(3),         # Values to display (rounded if needed)
    textposition="outside",                 # "inside" or "outside"

)

fig.update_layout(
    # title="Historical Electricity Cost in Germany (2007-2024)",
    xaxis_title="Year",
    yaxis_title="Electricity Price [€/kWh]",
    yaxis=dict(range=[0, 0.26]),
)

fig.update_xaxes(
    tickmode="linear",
    tickangle=-45
)

fig = apply_fig_defaults(fig, gridX=False)

export_fig(fig, "Figure_1_electricity_costs_germany")

fig.show()


Figure saved to out/paper-plots/Figure_1_electricity_costs_germany. width=1500, height=800.


In [16]:
# Figure 2: Relative share of energy sources in TWh in Germany for 1990 to 2023
# Excel: git:/out/gridmix/DF_3_RELATIVE.xlsx

fig2_df = sheets_dict["Figure 2"]
fig2_df = fig2_df.set_index("year")
fig2_df.index = fig2_df.index.astype(int)
fig2_df["percentage"] = (
    fig2_df["percentage"].astype(float) * 100
)  # Ensure percentage is float
# fig2_df["percentage"] = fig2_df["percentage"] * 100
print(fig2_df.head())

# fig = go.Figure()
unique_cats = fig2_df["Category"].unique()

# fig.add_hline

# symbol_map=SYMBOL_MAP,
fig = px.line(
    fig2_df,
    y="percentage",
    color="Category",
    symbol="Category",
    # color_discrete_sequence=get_color_sequence(unique_cats),
    color_discrete_map=COLOR_MAP,#get_color_sequence(unique_cats),
    # line_dash_map=LINE_DASH_MAP,
    labels={
        "year": "Year",
        "percentage": "Percentage [%]",
        "Category": "Energy Source"
    },
    # title="Energy Source Share in Germany" 
)

fig.update_xaxes(tickmode="linear", tickangle=-45)
fig.update_traces(marker=dict(size=14), line=dict(width=4))  # ← Hier Liniendicke setzen

fig = apply_fig_defaults(fig)
export_fig(fig, "Figure_2_historical_gridmix")
fig.show()

fig2_B_df = fig2_df.loc[(fig2_df.index >= 2007) & (fig2_df.index <= 2023)]

# Use only the present categories for color mapping
unique_cats_B = fig2_B_df["Category"].unique()

fig2 = px.line(
    fig2_B_df,
    y="percentage",
    color="Category",
    symbol="Category",

    # color_discrete_sequence=get_color_sequence(unique_cats_B),
    color_discrete_map=COLOR_MAP,
    labels={
        "year": "Year",
        "percentage": "Share [%]",
        "Category": "Energy Source"
    },
    # title="Energy Source Share in Germany" 
)

fig2.update_xaxes(tickmode="linear", tickangle=-45)
fig2 = apply_fig_defaults(fig2)
fig2.update_traces(marker=dict(size=14), line=dict(width=4))  # ← Hier Liniendicke setzen

export_fig(fig2, "Figure_2_historical_gridmix_2007_2023")
fig2.show()

         Category unit  percentage
year                              
1990    Hard coal    %       25.59
1990   Hydropower    %        3.53
1990      Lignite    %       31.06
1990  Mineral oil    %        1.96
1990  Natural gas    %        6.52
Figure saved to out/paper-plots/Figure_2_historical_gridmix. width=1500, height=800.


Figure saved to out/paper-plots/Figure_2_historical_gridmix_2007_2023. width=1500, height=800.


In [6]:
# Figure 3: Environmental impact per 1 kWh of electricity from different energy sources across multiple impact categories
# Excel: teams:LCA_Results+RawData.xlsx (1kWh LCA)

fig3_df = sheets_dict["Figure 3"]
fig3_df =fig3_df.rename(columns={fig3_df.columns[0]: "Impact",fig3_df.columns[1]: "Unit"})
energy_sources = fig3_df.columns[2:]  # all the energy source columns

# Melt to long-form for Plotly
df_melt = fig3_df.melt(
    id_vars=["Impact", "Unit"],
    value_vars=energy_sources,
    var_name="Source",
    value_name="Value"
)
# Compute group sums per indicator
sums = df_melt.groupby("Impact")["Value"].transform("sum")
df_melt["Share"] = df_melt["Value"] / sums

# Add label to show the unit as well
df_melt["ImpactLabel"] = df_melt["Impact"] + " [" + df_melt["Unit"] + "]"

print(df_melt.head())

unique_cats = df_melt["Source"].unique()

# Figure 3 with patterns
fig = go.Figure()

for i, source in enumerate(unique_cats):
    cat_data = df_melt[df_melt["Source"] == source]
    fig.add_bar(
        y=cat_data["ImpactLabel"],
        x=cat_data["Share"],
        name=source,
        orientation="h",
        marker=dict(
            color=COLOR_MAP.get(source, "#888"),
            pattern=dict(
                shape=PATTERN_MAP.get(source, ""),  
                fgcolor="rgba(50,50,50,0.7)"               # dark overlay for patterns
            ),
        ),
    )

fig.update_layout(
    barmode="stack",
    xaxis_tickformat=".0%",
    xaxis=dict(
        range=[0, 1.0],
        title="Share (%)",
        tickformat=".0%"
    ),
    yaxis_title="",  # Already in ImpactLabel
)
fig = apply_fig_defaults(fig)
fig.show()
export_fig(fig, "Figure_3_environmental_impact_per_kWh")

# Simple figure without patterns
# fig = px.bar(
#     df_melt,
#     x="Share",
#     y="ImpactLabel",
#     color="Source",
#     orientation="h",
#     color_discrete_sequence=get_color_sequence(unique_cats),
#     hover_data=["Unit"],
#     labels={
#         "Share": "",
#         "ImpactLabel": "",
#         "Source": "Energy Source"
#     },
#     title="Environmental Impact per kWh of Electricity from Different Energy Sources",
# )

# fig.update_layout(
#     barmode="stack",
#     xaxis_tickformat=".0%",
#     xaxis=dict(
#         range=[0, 1.0],
#         tickformat=".0%"
#     )
# )

# # Apply your styling defaults (legend below, font, etc.)
# fig = apply_fig_defaults(fig)

# fig.show()
# export_fig(fig, "Figure_3_environmental_impact_per_kWh")

   Impact           Unit         Source     Value     Share  \
0      AP  mol of H+-eq.  Photovoltaics  0.000651  0.051162   
1     GWP     kg CO2-eq.  Photovoltaics  0.099217  0.028380   
2   EP fw     kg PO4-eq.  Photovoltaics  0.000049  0.013965   
3   EP sw       kg N-eq.  Photovoltaics  0.000119  0.042970   
4  EP ter      mol N-eq.  Photovoltaics  0.001168  0.047603   

          ImpactLabel  
0  AP [mol of H+-eq.]  
1    GWP [kg CO2-eq.]  
2  EP fw [kg PO4-eq.]  
3    EP sw [kg N-eq.]  
4  EP ter [mol N-eq.]  


Figure saved to out/paper-plots/Figure_3_environmental_impact_per_kWh. width=1500, height=800.


In [7]:
# Figure 4: Environmental impacts of electricity consumption to produce 1 kg of hydrogen via AWE in Germany from 2007 to 2023: 
#        (a) with all environmental impacts and 
#        (b) without global warming potential.
# Excel: teams:LCA_Results+RawData.xlsx (1kg H2 LCA)

fig4_df = sheets_dict["Figure 4"]
fig4_df["year"] = fig4_df["year"].astype(int)
print(fig4_df.head())

# Map impact categories 
cols_all = [col for col in fig4_df.columns if col != "year"]

# Melt the DataFrame to long format for Plotly
fig4_a_df = fig4_df.melt(
    id_vars=["year"],
    value_vars=cols_all,
    var_name="Impact",
    value_name="Value"
)

fig4a = px.area(
    fig4_a_df,
    x="year",
    y="Value",
    color="Impact",
    color_discrete_sequence=COLOR_THEME,
    labels={
        "year": "Year",
        "Value": "Impact Value",
        # "Impact": "Impact Category"
    },
    # title="All Environmental Impacts"
)
fig4a.update_xaxes(tickmode="linear", tickangle=-45)
fig4a = apply_fig_defaults(fig4a)
fig4a.update_layout(
        width=DEFAULT_WIDTH,
        height=DEFAULT_HEIGHT,
)
fig4a.show()
export_fig(fig4a, "Figure_4a_env_impacts_incl_GWP", w=1200, h=720)

# Exclude GWP column
cols_without_gwp = [col for col in fig4_df.columns if col not in ("year", "GWP kg CO2-eqv.")]

# Melt the DataFrame to long format for Plotly
fig4_b_df = fig4_df.melt(
    id_vars=["year"],
    value_vars=cols_without_gwp,
    var_name="Impact",
    value_name="Value"
)

fig4b = px.area(
    fig4_b_df,
    x="year",
    y="Value",
    color="Impact",
    color_discrete_sequence=COLOR_THEME,
    labels={
        "year": "Year",
        "Value": "Impact Value",
        # "Impact": "Impact Category"
    },
    # title="Environmental Impacts (excluding GWP)"
)
fig4b.update_xaxes(tickmode="linear", tickangle=-45)
fig4b = apply_fig_defaults(fig4b)
fig4b.show()
export_fig(fig4b, "Figure_4b_env_impacts_excl_GWP", w=1200, h=720)


   year  AP Mole of H+-eqv.  EP fw kg P-eqv.  EP sw kg N-eqv.  \
0  2007            0.074234         0.045118         0.024243   
1  2008            0.070392         0.043147         0.023246   
2  2009            0.070279         0.044275         0.023371   
3  2010            0.069344         0.042233         0.022945   
4  2011            0.071040         0.044481         0.023691   

   EP ter Mole of N-eqv.  ODP kg CFC-11-eqv.  PM kg PM2.5-eqv.  \
0               0.169193            0.000001          0.004408   
1               0.162271            0.000001          0.004239   
2               0.161843            0.000001          0.004342   
3               0.162080            0.000001          0.004259   
4               0.166985            0.000001          0.004375   

   POCP kg NMVOC-eqv.  ADP kg Sb-eqv.  GWP kg CO2-eqv.  
0            0.044102        0.000030        34.363264  
1            0.042448        0.000031        32.942313  
2            0.042150        0.000032    

Figure saved to out/paper-plots/Figure_4a_env_impacts_incl_GWP. width=1500, height=800.


Figure saved to out/paper-plots/Figure_4b_env_impacts_excl_GWP. width=1500, height=800.


In [8]:
# Figure 5: Historical electricity costs associated with the production of 1 kg of hydrogen in Ger-many (2007–2023).
# Excel: Cost_Results+RawData (1kg H2 EnergyCost)

fig5_df = sheets_dict["Figure 5"]
print(fig5_df)
fig5_df = fig5_df.set_index("year")
fig5_df = fig5_df.sort_index(ascending=True)

fig = go.Figure()
fig.add_bar(
    x=fig5_df.index,
    y=fig5_df["price"],
    name="Electricity Cost per kg H2 [€/kg H2]",
    marker_color=COLOR_THEME[7],  # Use the same color as in Figure 1
    text=fig5_df["price"].round(2),
    textposition="outside",
)

fig.update_layout(
    # title="Energy Cost per kg H2",
    xaxis_title="Year",
    yaxis_title="Price [€/kg H2]",
    # yaxis=dict(range=[0, 0.26]),
)

fig.update_xaxes(tickmode="linear", tickangle=-45)

fig = apply_fig_defaults(fig)

export_fig(fig, "Figure_5_hydrogen_energy_costs_germany")

fig.show()

    year      price
0   2007   5.702620
1   2008   6.432965
2   2009   6.645870
3   2010   6.710550
4   2011   7.702310
5   2012   7.688835
6   2013   8.176630
7   2014   8.281735
8   2015   8.152375
9   2016   7.354655
10  2017   6.947710
11  2018   6.562325
12  2019   7.378910
13  2020   8.675205
14  2021   9.216900
15  2022  12.709620
16  2023  12.615295
Figure saved to out/paper-plots/Figure_5_hydrogen_energy_costs_germany. width=1500, height=800.


In [9]:
# Figure 6: Electricity Price Components for Industry in Germany from 2014 to 2023, adapted from [21].
FIG6_PATTERN_MAP = {
    "Concession levy": "/",
    "Renewable Energy Surcharge": "\\",
    "Combined Heat and Power Surcharge": "x",
    "§19 StromNEV Surcharge": "-",
    "Offshore Grid Surcharge": "|",
    "Interruptible Loads Surcharge": "+",
    "Electricity Tax": ".",
}

# Data Import
fig6_df = pd.read_csv(FIG_6_DATA)

# Melt the DataFrame to long format
categories = [col for col in fig6_df.columns if col != "years"]
df_melt = fig6_df.melt(id_vars=["years"], value_vars=categories, var_name="Category", value_name="Value")

# Normalize so every year's components sum to 1 (100%)
df_melt["Share"] = df_melt.groupby("years")["Value"].transform(lambda x: x / x.sum())

# Figure Plot

fig = go.Figure()

for i, category in enumerate(categories):
    cat_data = df_melt[df_melt["Category"] == category]
    fig.add_bar(
        y=cat_data["years"],
        x=cat_data["Share"],
        name=category,
        orientation="h",
        marker=dict(
            color=COLOR_THEME[i % len(COLOR_THEME)],
            pattern=dict(
                shape=FIG6_PATTERN_MAP.get(category, ""),  # fallback: no pattern
                fgcolor="rgba(50,50,50,0.7)"               # dark overlay for patterns
            )
        ),
    )

fig.update_layout(
    barmode="stack",
    xaxis_tickformat=".0%",
    xaxis=dict(
        range=[0, 1.0],
        tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
        title="Share (%)",
        tickformat=".0%"
    ),
    yaxis_title="Year",
    # title="Electricity Price Composition in Germany",
    plot_bgcolor="white",
)
fig = apply_fig_defaults(fig)
fig.show()
export_fig(fig, "Figure_6_electricity_price_components_2014_2023")


# Simple plot without patterns
# export_fig(fig, "Figure_6_electricity_price_components_2014_2023")

# fig = px.bar(
#     df_melt,
#     y="years",
#     x="Share",
#     color="Category",
#     color_discrete_sequence=COLOR_THEME,
#     orientation="h",
#     labels={
#         "years": "Year",
#         "Share": "Share (%)",
#         "Category": "Component"
#     },
#     title="Electricity Price Composition in Germany"
# )

# fig.update_layout(
#     barmode="stack",
#     xaxis_tickformat=".0%",
#     xaxis=dict(
#         range=[0, 1.0],
#         tickformat=".0%"
#     ),
# )

# fig.update_yaxes(
#     tickmode="linear",
# )

# fig = apply_fig_defaults(fig)
# fig.show()
# export_fig(fig, "Figure_6_electricity_price_components_2014_2023")


Figure saved to out/paper-plots/Figure_6_electricity_price_components_2014_2023. width=1500, height=800.


In [10]:
# Figure 7: Pareto optimization of grid mix scenarios (2025-2045) for cost and GWP per kg of H2
# Shreyas

In [11]:
# Figure 8. Clustered pareto-optimal grid mix scenarios (2025-2045) for cost and GWP per kg H2
# Shreyas